# Imports

In [1]:
import pandas as pd
from retriever import retrieve
from tqdm import tqdm
import os
from dotenv import load_dotenv

In [7]:
load_dotenv()
GROQ_API_KEYS=os.getenv("GROQ_API_KEYS")
#GROQ_API_KEYS

In [ ]:
groqs=GROQ_API_KEYS.split(',')
groqs

# TeleQnA

In [5]:
df = pd.read_json(r"E:\Vignesh\VIT\Project 1\TelecomRAG\Datasets\Telecom\teleqna_3gpp_questions.json")
df.head()

,question_id,question,option 1,option 2,option 3,option 4,category
0,question 0,What is the purpose of the Nmfaf_3daDataManage...,To configure the MFAF to map data or analytics...,To configure the MFAF to stop mapping data or ...,To supply data or analytics from the MFAF to n...,To fetch data or analytics from the MFAF based...,Standards specifications
1,question 4,How does a supporting UE attach to the same co...,It requests the core network node to remember ...,It uses information stored in the UE when it w...,It relies on the SIM/USIM card for information.,It performs a fresh attach procedure.,Standards specifications
2,question 11,When can the setting of the Privacy exception ...,Never,Only during emergency services,Anytime,Only with operator permission,Standards specifications
3,question 12,What does the NEF notify to the AF after deter...,AF Identifier,EAS address information,DNN,DNAI(s) or the updated DNAI information,Standards specifications
4,question 14,"In online charging, how are chargeable events ...",By the CTF,By the OCF,By the CGF,By the CHF,Standards specifications


In [ ]:
result=df.sample(10)
preds=[]
cts=[]
gr_idx=0
for row_num,row in tqdm(result.iterrows(),total=len(result)):
    question=row['question']
    options=[row[f"option {i}"] for i in range(1,5)]
    context_limit=5
    answer=None
    context=None
    while context_limit>=1:
        try:
            answer, context=retrieve(question,context_limit,options,groqs[gr_idx])
            break
        except Exception as e:
            error_msg=str(e).lower()
            token_error=any(
                term in error_msg
                for term in [
                    "token limit",
                    "maximum context length",
                    "context length exceeded",
                    "too many tokens",
                    "prompt is too long",
                    "request too large",
                    "413"
                ]
            )
            if token_error and context_limit>1:
                context_limit-=1
                continue

            if not token_error and gr_idx+ 1<len(groqs):
                gr_idx+=1
                continue
            break

    preds.append(answer if answer is not None else "0")
    cts.append(context if context is not None else "")
print(preds[:20])

 80%|████████  | 8/10 [02:01<00:39, 19.93s/it]

In [7]:
result=result.iloc[:8]
result['predictions']=preds[:8]
print(result.head())
result.to_csv('../output/teleqna_results.csv')
#print(df.iloc[168]['question'])

        question_id                                           question  \
554   question 3837  What is the purpose of providing an ID in NAS ...   
1305  question 8774  What does the 'Request-response' communication...   
766   question 5275  What does MnS component type B refer to? [3GPP...   
1301  question 8755  What is the purpose of the Internal-Group Iden...   
849   question 5826  What happens when an MPS session request canno...   

                                               option 1  \
554       To assign UE capabilities by the manufacturer   
1305  Streaming management data over an established ...   
766                Generic operations and notifications   
1301         To support user confidentiality protection   
849                      The session request is dropped   

                                               option 2  \
554                       To reduce signalling overhead   
1305  Requesting a subscription to receive network e...   
766   Performance infor

In [8]:
ans_df = pd.read_json(r"E:\Vignesh\VIT\Project 1\TelecomRAG\Datasets\Telecom\teleqna_3gpp_answers.json")
ans_df.head()

,question_id,answer,explanation,category
0,question 0,2,The Nmfaf_3daDataManagement_Deconfigure servic...,Standards specifications
1,question 4,2,A supporting UE in a shared network attaches t...,Standards specifications
2,question 11,3,The setting of the Privacy exception list can ...,Standards specifications
3,question 12,4,"After determining the suitable DNAI(s), NEF no...",Standards specifications
4,question 14,2,The CTF transforms each chargeable event into ...,Standards specifications


In [12]:
eval_df=pd.merge(result,ans_df,how='inner',on='question_id')[['question_id', 'question', 'option 1', 'option 2', 'option 3','option 4', 'predictions','answer']]
eval_df=eval_df[eval_df['predictions']!=""]
eval_df.head(10)

,question_id,question,option 1,option 2,option 3,option 4,predictions,answer
0,question 3837,What is the purpose of providing an ID in NAS ...,To assign UE capabilities by the manufacturer,To reduce signalling overhead,To store UE Radio Capabilities in the AMF,"To request UE capabilities for NR, EUTRA, UTRA...",2,2
1,question 8774,What does the 'Request-response' communication...,Streaming management data over an established ...,Requesting a subscription to receive network e...,Authenticating the management service consumer,Invoking an operation and providing a response,4,4
2,question 5275,What does MnS component type B refer to? [3GPP...,Generic operations and notifications,Performance information of the managed entity ...,Management information represented by informat...,Management services provided by an MnS producer,4,3
3,question 8755,What is the purpose of the Internal-Group Iden...,To support user confidentiality protection,To enable roaming scenarios,To uniquely identify the UE within the AMF,To identify the user equipment accessing the 5...,4,5
4,question 5826,What happens when an MPS session request canno...,The session request is dropped,The session request is terminated,The session request is queued,The session request is ignored,3,3
5,question 2268,What physical layer measurement quantity can b...,RSRP,RSRQ,RSSI,CQI,3,3
6,question 5067,In what market sector are 3GPP networks expect...,Global revenues,AV content production,Mass audiences,Mobile consumption,4,2
7,question 5982,What capabilities should a Shared E-UTRAN/NG-R...,Differentiate traffic associated with individu...,Conduct admission control based on allocated r...,Provide monitoring of resource usage,All of the above,4,4


In [10]:
eval_df['answer']=eval_df['answer'].astype(int)
eval_df['predictions']=eval_df['predictions'].astype(int)

In [11]:
len(eval_df[eval_df['answer']==eval_df['predictions']])*100/len(eval_df)

62.5